### Production CNN Model ensemble on the test (holdout data) scale images from AFSC, ADFG, DFO, and NSRAA

Each model, (1) EfficientNetV2L, (2) NASNetLarge, and (3) ConvNeXtLrg were trained independently on the training and validation preprocessed SAM datasets, which were from the same agencies. The SAM image data were randomly split into the Training (n=10916), Validation (n=2343), and Testing (n=2345), but were first stratified by total age in order to keep the age class distribution even between the datasets. Each dataset were storage on disk within separate directories

This script, imports raw images, converts to SAM images and other preprocesses steps. and then the models are loaded from disk, both their weights and architectures, and fit to the SAM data. The model predictions were pooled based on their performance on the validation data during training. 




Date: 05/22/2025

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  

from PIL import Image
import numpy as np
import pandas as pd
import tensorrt
import io
import cv2
import matplotlib.image as mpimg
#import cx_Oracle 
import numpy.random as nr
import matplotlib.pyplot as plt
#import seaborn as sns
import tensorflow as tf
from skimage.transform import resize as sk_resize
from pathlib import Path
import torch
from Scalevision.per_segment_anything import SamPredictor, sam_model_registry
import keras
import keras.layers as layers
from keras import optimizers
import time
import os, shutil
from torch.nn import functional as F
#from skimage.transform import resize
from typing import Tuple

%matplotlib inline

2026-02-09 11:55:11.657857: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
start = time.time()

In [ ]:
# Enables memory growth instead of allocating all the memory at once. 
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    print(e)

#### Segmenting raw scale images using the Segment Anything Model (SAM)

In [ ]:
# Define the path to the model files
model_path = Path('/opt/chumputer/Scalevision/crop_vision')
file_names = ['mask_weights.pth', 'target_feats.pth', 'weights.pth']

# Create a dictionary of file paths for each model component
file_paths = {file_name: model_path / file_name for file_name in file_names}

# Load the model if all required files exist
if all(path.exists() for path in file_paths.values()):
    print('Loading existing model')
    loaded_files = {
        file_name: torch.load(path) for file_name, path in file_paths.items()
    }

In [ ]:
class Mask_Weights(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = torch.nn.Parameter(torch.ones(2, 1, requires_grad=True) / 3)

    def forward(self, x):
        return x * self.weights


def load_trained_model(device='cuda'):
    model_path = Path('/opt/chumputer/Scalevision/crop_vision')
    file_names = [
        'mask_weights.pth',
        'target_feats.pth',
        'weights.pth',
    ]

    # Create a dictionary of file paths for each model component
    file_paths = {file_name: model_path / file_name for file_name in file_names}

    # Load the base SAM model
    sam_ckpt = model_path / 'sam_vit_h_4b8939.pth'
    sam = sam_model_registry['vit_h'](checkpoint=sam_ckpt).to(device)

    # Initialize the predictor
    predictor = SamPredictor(sam)

    # Load mask weights
    mask_weights = Mask_Weights().to(device)
    mask_weights.load_state_dict(
        torch.load(file_paths['mask_weights.pth'], map_location=device)
    )

    # Load target features and weights
    target_feats = torch.load(file_paths['target_feats.pth'], map_location=device)
    weights = torch.load(file_paths['weights.pth'], map_location=device)

    weights_np = weights.detach().cpu().numpy()

    return predictor, mask_weights, target_feats, weights, weights_np


predictor, mask_weights, target_feats, weights, weights_np = load_trained_model()

In [ ]:
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

def get_bounding_box(binary_mask: np.ndarray):
    """
    Calculate the bounding box coordinates of a binary mask.

    This function takes a 2D binary mask (where True or 1 represents the object,
    and False or 0 represents the background) and finds the smallest rectangle
    that encompasses all the True values in the mask.

    Args:
        binary_mask (np.ndarray): 2D numpy array representing the binary mask.
            True or 1 indicates the object, False or 0 indicates the background.

    Returns:
        tuple[int, int, int, int]: A tuple containing the coordinates of the
            bounding box in the format (min_x, min_y, max_x, max_y).
            - min_x: leftmost x-coordinate of the object
            - min_y: topmost y-coordinate of the object
            - max_x: rightmost x-coordinate of the object
            - max_y: bottommost y-coordinate of the object
    """
    # np.where returns two arrays: y_coords and x_coords
    # Each array contains the indices where binary_mask is True
    y_coords, x_coords = np.where(binary_mask)

    # Find the minimum and maximum values for x and y coordinates
    # This effectively gives us the corners of the bounding box
    return min(x_coords), min(y_coords), max(x_coords), max(y_coords)


def crop_to_mask(input_image: Image, binary_mask: np.ndarray) -> Image:
    # Convert binary mask to PIL Image and get its bounding box
    mask_image = Image.fromarray(binary_mask)
    bounding_box = get_bounding_box(binary_mask)

    # Crop input image and mask to the bounding box
    cropped_input = input_image.crop(bounding_box)
    cropped_mask = mask_image.crop(bounding_box)

    # Create grayscale mask and black background
    grayscale_mask = cropped_mask.convert('L')
    background = Image.new('L', cropped_input.size)

    # Apply mask to the cropped input to create final image
    masked_result = Image.composite(
        cropped_input.convert('L'), background, grayscale_mask
    )

    return masked_result.convert('RGB')


def point_selection(
    mask_sim: torch.Tensor, topk: int = 1
) -> Tuple[np.ndarray, np.ndarray]:
    # Top-1 point selection
    w, h = mask_sim.shape
    topk_xy = mask_sim.flatten(0).topk(topk)[1]
    topk_x = (topk_xy // h).unsqueeze(0)
    topk_y = topk_xy - topk_x * h
    topk_xy = torch.cat((topk_y, topk_x), dim=0).permute(1, 0)
    topk_label = np.array([1] * topk)
    topk_xy = topk_xy.cpu().numpy()

    return topk_xy, topk_label


def get_image_mask(image_blob: bytes) -> np.ndarray:
    size = 1024
    # Load test image
    test_image = cv2.imdecode(
        np.frombuffer(image_blob, dtype=np.uint8), cv2.IMREAD_UNCHANGED
    )
    test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
    test_image = cv2.resize(test_image, (size, size))

    # Image feature encoding
    predictor.set_image(test_image)
    test_feat = predictor.features.squeeze()

    # Cosine similarity
    C, h, w = test_feat.shape
    test_feat = test_feat / test_feat.norm(dim=0, keepdim=True)
    test_feat = test_feat.reshape(C, h * w)

    k = 5
    sim_list = [targ_feat @ test_feat for targ_feat in target_feats]
    top_k_sim_list = []
    for sim in sim_list:
        top_k_values, _ = torch.topk(sim.view(-1), k)
        sim[sim < top_k_values[-1]] = 0  # Replace values not in the top-k with 0
        top_k_sim_list.append(sim)
    sim = sum(top_k_sim_list) / len(
        top_k_sim_list
    )  # Average similarity of top-k values

    sim = sim.reshape(1, 1, h, w)
    sim = F.interpolate(sim, scale_factor=4, mode='bilinear')
    sim = predictor.model.postprocess_masks(
        sim, input_size=predictor.input_size, original_size=predictor.original_size
    ).squeeze()

    # Positive location prior
    topk_xy, topk_label = point_selection(sim, topk=1)

    # First-step prediction
    masks, scores, logits, logits_high_list = predictor.predict(
        point_coords=topk_xy, point_labels=topk_label, multimask_output=True
    )

    # Weighted sum three-scale masks
    weights_list = [
        logit_high * weight.unsqueeze(-1)
        for logit_high, weight in zip(logits_high_list, weights)
    ]
    logit_high = sum(weights_list) / len(weights_list)  # Average weighted sum
    mask = (logit_high > 0).detach().cpu().numpy()

    logits_list = [
        logit * weight_np[..., None] for logit, weight_np in zip(logits, weights_np)
    ]
    logit = sum(logits_list) / len(logits_list)  # Average logits

    # Cascaded Post-refinement-1
    y, x = np.nonzero(mask)
    if y.size == 0 or x.size == 0:
        # handle the case when there are no non-zero values in the mask
        print('Mask has no non-zero values.')
    else:
        x_min = x.min()
        x_max = x.max()
        y_min = y.min()
        y_max = y.max()
    input_box = np.array([x_min, y_min, x_max, y_max])
    masks, scores, logits, _ = predictor.predict(
        point_coords=topk_xy,
        point_labels=topk_label,
        box=input_box[None, :],
        mask_input=logit[None, :, :],
        multimask_output=True,
    )
    best_idx = np.argmax(scores)

    # Cascaded Post-refinement-2
    y, x = np.nonzero(masks[best_idx])
    x_min = x.min()
    x_max = x.max()
    y_min = y.min()
    y_max = y.max()
    input_box = np.array([x_min, y_min, x_max, y_max])
    masks, scores, logits, _ = predictor.predict(
        point_coords=topk_xy,
        point_labels=topk_label,
        box=input_box[None, :],
        mask_input=logits[best_idx : best_idx + 1, :, :],
        multimask_output=True,
    )
    best_idx = np.argmax(scores)

    return masks[best_idx]


def image_to_bytes(image: Image) -> bytes:
    buffer = io.BytesIO()
    image.save(buffer, format='PNG')
    return buffer.getvalue()

In [ ]:
# from pympler import muppy, summary

# all_objects = muppy.get_objects()
# sum1 = summary.summarize(all_objects)
# summary.print_(sum1)

In [ ]:
import os
import numpy as np
from PIL import Image
from skimage.transform import resize as sk_resize
from pathlib import Path

# Define the base path for the raw input images
base_folder = '/opt/chumputer/scale_images/Image_raw_code_testing_good/Testing'

def process_image(original_image: Image) -> Image:
    image_bytes = image_to_bytes(original_image)
    mask: np.ndarray = get_image_mask(image_bytes)

    resized_mask: np.ndarray = sk_resize(
        mask,
        (original_image.size[1], original_image.size[0]),
        anti_aliasing=False,
        mode='edge',
    )

    masked_image: Image = crop_to_mask(original_image, resized_mask)
    return masked_image

def classify_and_store_images():
    sam_images_in_memory = []

    for file_name in os.listdir(base_folder):
        if file_name.lower().endswith(('.png', '.tif', '.tiff', '.jpg', '.jpeg')):
            image_path = os.path.join(base_folder, file_name)
            try:
                with Image.open(image_path) as img:
                    processed_image = process_image(img)
                    sam_images_in_memory.append((file_name, processed_image))

                    print(f"Processed {file_name} and stored in memory")

            except Exception as e:
                print(f"Error processing {file_name}: {e}")
    
    return sam_images_in_memory

if __name__ == "__main__":
    sam_images = classify_and_store_images()


In [ ]:
num_images = min(4, len(sam_images))  

plt.figure(figsize=(12, 6))
for i in range(num_images):
    file_name, image = sam_images[i]
    plt.subplot(1, num_images, i + 1)
    plt.imshow(image)
    plt.title(file_name)
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
for file_name, image in sam_images[:4]:  
    print(f"{file_name}: {image.size} (width, height)")

In [ ]:
# loop through image in memory and extracts the age class label from the end of the file name 
# and create a pandas dataframe that includes the filename and label (total age). 

def load_images_from_memory(image_list):
    labels = ['_1', '_2', '_3', '_4', '_5', '_6', '_7']
    files_by_label = {label: [] for label in labels}
    
    for file_name, image in image_list:
        
        for label in labels:
            if file_name.lower().endswith(label + '.png') or \
               file_name.lower().endswith(label + '.jpg') or \
               file_name.lower().endswith(label + '.jpeg') or \
               file_name.lower().endswith(label + '.tif') or \
               file_name.lower().endswith(label + '.tiff'):
                files_by_label[label].append(file_name)

    data = [(file, label) for label, files in files_by_label.items() for file in files]
    return pd.DataFrame(data, columns=['filename', 'label'])

df_test = load_images_from_memory(sam_images)
print(f"Total images in test set: {len(df_test)}")


In [ ]:
# this simply removes the underscore that was used to separate file name from the age value
df_test['label'] = df_test['label'].str.replace(r'_(\d+)', r'\1', regex=True)
df_test['label'] = pd.to_numeric(df_test['label'], errors='coerce')

In [ ]:
df_test.tail()

In [ ]:
# separating the file name and age label into two separate lists but they maintain the same index
test_images_link = df_test['filename'].tolist()
test_labels = df_test['label'].tolist()

In [ ]:
# show distribution by age class
from collections import Counter

count_data = Counter(test_labels)

x = list(count_data.keys())
y = list(count_data.values())

plt.bar(x, y, color='skyblue', edgecolor='black')

plt.xlabel('Value')
plt.ylabel('Frequency') 
plt.title('Validation Data: Total Age Frequency')

plt.tight_layout()
#plt.savefig('ValidationAgeLabels.pdf') 
plt.show()

In [ ]:
# This is a key cell block, it links the file name from the image list
# Those images are resized and then into a dict. 
# The index is maintained from the test_images_link in order to link them back to the age labels 
# This step is very efficient if the computer has sufficient memory otherwise batch processing would be necessary.

image_resize = (512, 512)

def resize_images_in_place(sam_images_list, target_size=image_resize):
    for i, (fname, img) in enumerate(sam_images_list):
        resized_img = img.resize(target_size, Image.LANCZOS)
        sam_images_list[i] = (fname, resized_img)  

def load_images_from_memory(image_filenames, sam_images_list):
    images = {}
    # Convert to dict for faster lookup
    sam_dict = {fname: img for fname, img in sam_images_list}
    
    for filename in image_filenames:
        if filename in sam_dict:
            images[filename] = sam_dict[filename]
        else:
            print(f"{filename} not found in in-memory image list.")
    
    return images

# First, resize all images in-place
resize_images_in_place(sam_images)

# Then load the resized images you want
test_images = load_images_from_memory(test_images_link, sam_images)


In [ ]:
list(test_images.items())[0:4]

In [ ]:
# This block creates two functions to make sure all images have 3-channe, ie are color images, and if not to covert them regardless of the image imput type. 
# Important: The Keras models used in this project require all input images to be in color, have 3-channels

import cv2
from tqdm import tqdm

def process_image(img):

    if not isinstance(img, Image.Image):
        raise TypeError("Input has to be a PIL.Image.Image object.")
    
    img_np = np.array(img)
    
    print(f"Processing image with shape: {img_np.shape}")
    
    if len(img_np.shape) == 2:
        img_rgb = cv2.cvtColor(img_np, cv2.COLOR_GRAY2RGB)
    elif len(img_np.shape) == 3:
        if img_np.shape[2] == 1:
            img_rgb = np.repeat(img_np, 3, axis=2)
        elif img_np.shape[2] == 3:
            img_rgb = img_np
        else:
            raise ValueError(f"Unexpected number of channels in the image: {img_np.shape[2]}")
    else:
        raise ValueError(f"Unexpected image dimensions: {img_np.shape}")
    
    return img_rgb

def process_images(image_dict):
    processed_images = []
    error_count = 0
    for filename, img in tqdm(image_dict.items(), desc="Processing Images"):
        try:
            processed_img = process_image(img)
            processed_images.append(processed_img)
        except (TypeError, ValueError) as e:
            print(f"Error processing {filename}: {e}")
            error_count += 1
    
    print(f"Number of errors: {error_count}")
    return processed_images    

In [ ]:
# This code running the above functions, creating the images to array and normalizing them. 
processed_test_images = process_images(test_images) 
processed_test_images = np.array(processed_test_images)
x_test = processed_test_images.astype("float32") / 255

for img in test_images.values():
     img.close()

In [ ]:
# Count of images in memory
print(x_test.shape[0], "test samples")

In [ ]:
# subtracting 1 from the labels. We don't have an zero total ages :-)
y_test_class = np.asarray([float(num) - 1 for num in test_labels])

In [ ]:
#import tensorflow as tf
tf.keras.backend.clear_session()

In [ ]:
# Checking available GPUs 
print("GPU available: ", tf.config.list_physical_devices('GPU'))

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications import ConvNeXtLarge
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import Constant

class LayerScale(tf.keras.layers.Layer):
    def __init__(self, init_values=1e-6, projection_dim=None, **kwargs):
        super().__init__(**kwargs)
        self.init_values = init_values
        self.projection_dim = projection_dim

    def build(self, input_shape):
        if self.projection_dim is None:
            if input_shape[-1] is None:
                raise ValueError("projection_dim must be set or inferable from input shape.")
            self.projection_dim = input_shape[-1]

        self.scale = self.add_weight(
            name='scale',
            shape=(self.projection_dim,),
            initializer=Constant(self.init_values),
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        return inputs * self.scale

    def get_config(self):
        config = super().get_config()
        config.update({
            'init_values': self.init_values,
            'projection_dim': self.projection_dim
        })
        return config

def build_model(input_shape=(512, 512, 3), dropout_rate=0.5, l2_reg=1e-4):
    pretrained_ConvNeXtLarge = ConvNeXtLarge(
        include_top=False,
        include_preprocessing=False,
       # weights='imagenet',
        input_shape=input_shape
    )

    # Optionally freeze part of the base
    # for layer in base_model.layers[:len(base_model.layers)//3]:
    #     layer.trainable = False

    x = GlobalAveragePooling2D()(pretrained_ConvNeXtLarge.output)
    x = Dense(4096, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    predictions = Dense(1, activation='linear')(x)

    ConvNeXtLargemodel = Model(inputs=pretrained_ConvNeXtLarge.input, outputs=predictions)    

    return ConvNeXtLargemodel

In [ ]:
strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    ConvNeXtLargemodel = build_model()
    ConvNeXtLargemodel.load_weights("/home/rames/SAM_models/ConvNeXtLarge_SAM_model_9102025_file.hdf5") #ConvNeXtLarge_SAM_model_file


In [ ]:
ConvNeXtLarge_model_predictions = ConvNeXtLargemodel.predict(x_test) 

In [ ]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test_class, ConvNeXtLarge_model_predictions)
print("MSE:", mse)

In [ ]:
tf.keras.backend.clear_session()

In [ ]:
# Loading the efficientNet model and applying mirrorstrategy to use both GPUs. 
import tensorflow as tf
from keras.models import load_model

strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    effnetv2l_model = load_model("/home/rames/SAM_models/effnetv2l_SAM_model_992025_file.hdf5" #  "/opt/chumputer/effnetv2l_SAM_model_file.hdf5")
    )

#model.summary()

In [ ]:
# fit model 
effnetv2l_predictions = effnetv2l_model.predict(x_test) 

In [ ]:
mse = mean_squared_error(y_test_class, effnetv2l_predictions)
print("MSE:", mse)

In [ ]:
tf.keras.backend.clear_session()

In [ ]:
# Loading the NASNetLarge model and applying mirrorstrategy to use both GPUs. 
import tensorflow as tf
from keras.models import load_model

strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    NASNetLarge_model = load_model("/home/rames/SAM_models/NASNetLarge_SAM_model_992025_file.hdf5"  #load_model("/opt/chumputer/NASNetLarge_SAM_model_file.hdf5")
    )

In [ ]:
NASNetLarge_predictions = NASNetLarge_model.predict(x_test) 

In [ ]:
mse = mean_squared_error(y_test_class, NASNetLarge_predictions)
print("MSE:", mse)

In [ ]:
# The history of the model training. 
# This history is necessary to extract the min MSE in order to weight the models based on their individual performance against the validation data
import pickle

with open('/home/rames/SAM_models/effnetv2l_SAM_992025_history.pkl', 'rb') as f:
    effnet_history = pickle.load(f)

In [ ]:
with open('/home/rames/SAM_models/NASNetLarge_992025_history.pkl', 'rb') as f:
    NASNet_history = pickle.load(f)

In [ ]:
with open('/home/rames/SAM_models/ConvNeXtLargeSAM_9102025history.pkl', 'rb') as f:
    ConvNeXt_history = pickle.load(f)

In [ ]:
effnet_train_mse = effnet_history['mse']
effnet_test_mse = effnet_history['val_mse']
#effnet_min_mse_epoch = epochs[effnet_test_mse.index(min(effnet_test_mse))]
effnet_min_val_mse = min(effnet_test_mse)

############################################# I didn't save the pkl file in this model run, but know the lowest MSE
#NASNet_train_mse = NASNet_history['mse']
#NASNet_test_mse = NASNet_history['val_mse']
#NASNet_min_mse_epoch = epochs[NASNet_test_mse.index(min(NASNet_test_mse))]
NASNet_min_val_mse = 0.0865 #min(NASNet_test_mse) 

ConvNeXt_train_mse = ConvNeXt_history['mse']
ConvNeXt_test_mse = ConvNeXt_history['val_mse']
# #ConvNeXt_min_mse_epoch = epochs[ConvNeXt_test_mse.index(min(ConvNeXt_test_mse))]
ConvNeXt_min_val_mse = min(ConvNeXt_test_mse)

In [ ]:
# the min MSE from each model

#print(f"Effnet v2L Minimum Validation MSE occurred at epoch {effnet_min_mse_epoch}")
print(f"Effnet v2L Minimum Validation MSE value: {effnet_min_val_mse:.4f}")

#print(f"NASNet Minimum Validation MSE occurred at epoch {NASNet_min_mse_epoch}")
print(f"NASNet Minimum Validation MSE value: {NASNet_min_val_mse:.4f}")

print(f"ConvNeXt Minimum Validation MSE value: {ConvNeXt_min_val_mse:.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, len(effnet_test_mse) + 1))

min_mse_epoch = epochs[effnet_test_mse.index(min(effnet_test_mse))]
min_val_mse = min(effnet_test_mse)

plt.plot(epochs, effnet_train_mse, label='Training MSE')
plt.plot(epochs, effnet_test_mse, label='Validation MSE')

plt.axvline(x=min_mse_epoch, color='black', linestyle='--', label=f'Min Validation MSE Epoch {min_mse_epoch}')
plt.scatter(min_mse_epoch, min_val_mse, color='orange', edgecolor='black', zorder=5)

plt.annotate(f'Min: {min_val_mse:.4f}',
             xy=(min_mse_epoch, min_val_mse),
             xytext=(min_mse_epoch + 0.5, min_val_mse - 0.04),  
             fontsize=10)

plt.title('EfficientNet v2L Mean Squared Error over Epochs from Training')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#import matplotlib.pyplot as plt

train_mse = NASNet_history['mse']
test_mse = NASNet_history['val_mse']

epochs = list(range(1, len(test_mse) + 1))

min_mse_epoch = epochs[test_mse.index(min(test_mse))]
min_val_mse = min(test_mse)

plt.plot(epochs, train_mse, label='Training MSE', color='green')
plt.plot(epochs, test_mse, label='Validation MSE', color='red')

plt.axvline(x=min_mse_epoch, color='black', linestyle='--', label=f'Min Validation MSE Epoch {min_mse_epoch}')
plt.scatter(min_mse_epoch, min_val_mse, color='red', edgecolor='black', zorder=5)

plt.annotate(f'Min: {min_val_mse:.4f}',
             xy=(min_mse_epoch, min_val_mse),
             xytext=(min_mse_epoch + 0.5, min_val_mse - 0.04),  
             fontsize=10)

plt.title('NASNetLarge Mean Squared Error over Epochs from Training')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.ylim(0,1)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

train_mse = ConvNeXt_history['mse']
test_mse = ConvNeXt_history['val_mse']

epochs = list(range(1, len(test_mse) + 1))

min_mse_epoch = epochs[test_mse.index(min(test_mse))]
min_val_mse = min(test_mse)

plt.plot(epochs, train_mse, label='Training MSE', color='cyan')
plt.plot(epochs, test_mse, label='Validation MSE', color='magenta')

plt.axvline(x=min_mse_epoch, color='black', linestyle='--', label=f'Min Validation MSE Epoch {min_mse_epoch}')
plt.scatter(min_mse_epoch, min_val_mse, color='magenta', edgecolor='black', zorder=5)

plt.annotate(f'Min: {min_val_mse:.4f}',
             xy=(min_mse_epoch, min_val_mse),
             xytext=(min_mse_epoch + 0.5, min_val_mse - 0.04),  
             fontsize=10)

plt.title('ConvNeXt Mean Squared Error over Epochs from Training')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.ylim(0,1)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Creating weights for each model inversely proportional to their validation MSE
# weighting each of the predictions to create final prediction

import numpy as np
from sklearn.metrics import mean_squared_error

mse1 = effnet_min_val_mse
mse2 = NASNet_min_val_mse
mse3 = ConvNeXt_min_val_mse

weights = np.array([1/mse1, 1/mse2, 1/mse3])
weights /= weights.sum()  

test_predictions1 = effnetv2l_predictions + 1
test_predictions2 = NASNetLarge_predictions + 1
test_predictions3 = ConvNeXtLarge_model_predictions + 1

predictions_round1 = np.round(test_predictions1)
predictions_round2 = np.round(test_predictions2)
predictions_round3 = np.round(test_predictions3)

combined_test_predictions = (
    weights[0] * test_predictions1 +
    weights[1] * test_predictions2 +
    weights[2] * test_predictions3
)

combined_test_predictions_round = np.round(combined_test_predictions)

In [ ]:
# combining data to create a dataframe that includes file name, labels, as well as the predictions. 
test_labels = np.array(test_labels)
combined_test_predictions = np.array(combined_test_predictions)
combined_test_predictions_round = np.array(combined_test_predictions_round)
test_predictions1 = np.array(test_predictions1)
test_predictions2 = np.array(test_predictions2)
test_predictions3 = np.array(test_predictions3)

In [ ]:
# output dataframe
pred = np.concatenate((test_labels.reshape(-1, 1), combined_test_predictions.reshape(-1, 1), test_predictions1.reshape(-1, 1), 
                       test_predictions2.reshape(-1, 1), test_predictions3.reshape(-1, 1), combined_test_predictions_round.reshape(-1, 1)), axis=1)  


In [ ]:
# dataframe headers
pred_df = pd.DataFrame(data=pred, columns=['y_test', 'combined_predictions', 'effnetv2l', 'NASNetLarge', 'ConvNeXt', 'combined_test_predictions_round']) 

In [ ]:
pred_df.head(3)

In [ ]:
# combined prediction's R^2 value
from sklearn.metrics import r2_score

r2 = r2_score(pred_df['y_test'], pred_df['combined_predictions'])

In [ ]:
round(r2,3)

In [ ]:
# Final prediction plot vs labeled ages

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

pred_df['y_test'] = pd.to_numeric(pred_df['y_test'], errors='coerce')

r2 = r2_score(pred_df['y_test'], pred_df['combined_predictions'])

plt.figure(figsize=(7, 5))

sns.violinplot(data=pred_df, x='y_test', y='combined_predictions', inner=None, linewidth=0, color='skyblue')

size = 3  
sns.stripplot(data=pred_df, x='y_test', y='combined_predictions', color='darkblue', alpha=0.15, size=size, jitter=0.02)

#sns.regplot(data=pred_df, x='y_test', y='combined_predictions', scatter=False, color='red', line_kws={"linewidth": 2})

plt.ylabel("Predicted Age (CNN Ensemble)", fontsize=14)
plt.xlabel("Labeled Age (years)", fontsize=12)
plt.title('ADFG Data: CNN Ensemble Predictions vs. Labeled Ages (n=4000)', fontsize=12)
plt.ylim(1, 7.5)

plt.text(x=0.05, y=0.93, s=f"$R^2$ = {r2:.3f}", fontsize=10, transform=plt.gca().transAxes)


plt.grid(True, which='major', linestyle='--', linewidth=0.5, color='gray', alpha=0.4)
plt.minorticks_on()
plt.grid(True, which='minor', linestyle=':', linewidth=0.3, color='gray', alpha=0.3)
plt.tight_layout()
#plt.savefig('ViolinPlotADFG4000.png', dpi=700)
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix

y_true = pred_df['y_test']
y_pred = pred_df['combined_test_predictions_round']

labels = [1, 2, 3, 4, 5, 6, 7]

conf_matrix = confusion_matrix(y_true, y_pred, labels=labels)

conf_df = pd.DataFrame(conf_matrix, index=labels, columns=labels)

print("Confusion Matrix:")
print(conf_df)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

y_true = pred_df['y_test']
y_pred = pred_df['combined_test_predictions_round']

labels = [1, 2, 3, 4, 5, 6, 7]

conf_matrix = confusion_matrix(y_true, y_pred, labels=labels)

conf_df = pd.DataFrame(conf_matrix, index=labels, columns=labels)

plt.figure(figsize=(7, 5))
sns.heatmap(conf_df, annot=True, fmt='d',  cmap='Blues', cbar=True, linewidths=0.5, xticklabels=labels, yticklabels=labels)

plt.title('Confusion Matrix', fontsize=16)
plt.xlabel('Predicted Labels', fontsize=14)
plt.ylabel('True Labels', fontsize=14)
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()
#plt.savefig('ConfusionMatrixADFG4000.png', dpi=700)
plt.show()


In [ ]:
overall_accuracy = np.trace(conf_matrix) / np.sum(conf_matrix)

print("Overall Accuracy: {:.2f}%".format(overall_accuracy * 100))